### DOUBLE DQN VS RANDOM

In [1]:
import time
import torch
from collections import Counter
from domain.configs import MAX_STEPS_PER_EPISODE, LOG_EVERY
from environment.grenight_environment import GrenightEnvironment
from agents.double_dqn_vs_random.agent import Agent

In [2]:
def play_game(env_arg: GrenightEnvironment,
              agent_arg: Agent) -> tuple[str, int, dict]:

    state = env_arg.reset()
    done = False
    move_count = 0
    acting_player_is_white = True
    reward = 0.0
    info = None

    while not done and move_count < MAX_STEPS_PER_EPISODE:
        acting_player_is_white = env_arg.is_white_on_turn
        if acting_player_is_white:
            mask = env_arg.action_mask()
            action = agent_arg.select_action(state, mask, 0)
        else:
            action = env_arg.sample()

        state, reward, done, info = env_arg.step(action)

        move_count += 1

    if not done:
        return "truncated", move_count, info

    if reward == 0:
        return "draw", move_count, info

    winner_is_white = acting_player_is_white if reward == 1 else not acting_player_is_white

    return ("white_win", move_count, info) if winner_is_white else ("black_win", move_count, info)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"using device: {device}")

for ep in (2500, 19408):
    env = GrenightEnvironment()
    outcomes_counter = Counter()
    draw_reasons_counter = Counter()

    agent = Agent(
        num_planes=env.state_encoder.NUM_PLANES,
        rows=5,
        columns=4,
        num_actions=env.action_encoder.NUM_ACTIONS,
        device=device
    )

    checkpoint = torch.load(f"../double_dqn_vs_random/checkpoints/ep{ep}.pt", map_location=device)

    agent.policy_net.load_state_dict(checkpoint["policy_state_dict"])
    agent.target_net.load_state_dict(checkpoint["target_state_dict"])
    agent.optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

    start_time = time.perf_counter()
    for _ in range(LOG_EVERY):
        outcome, steps, game_info = play_game(env, agent)
        outcomes_counter[outcome] += 1
        if game_info["draw_reason"] is not None:
            draw_reasons_counter[game_info["draw_reason"]] += 1
    end_time = time.perf_counter()

    print(f"STATS OUT FROM: {LOG_EVERY} GAMES IN: {ep} CHECKPOINT\n"
          f"Outcomes: {outcomes_counter}\n"
          f"Draw reasons: {draw_reasons_counter}\n"
          f"Execution time: {end_time - start_time:.2f} seconds\n")

using device: cpu
